# 予報の品質調査（2026-09-30）
公開済みの片道便を粒度として、コメント、往路・復路の比較、10/4の欠落、公開時刻を調査した。
16:00 JST公開分の10/4予定8便: AI生成4、AI失敗かつ気象/MLあり3、予測対象外1。
再実行した場合は次の公開更新によって結果が変わる。

In [ ]:
import json, urllib.request
url = 'https://tokai-kisen-forecast-preview.hareharenohara.workers.dev/api/days?from=2026-09-30'
with urllib.request.urlopen(url) as response:
    payload = json.load(response)
for day in payload['days']:
    rows = day['services']
    print(day['date'], len(rows), sum(r['ai_status']=='generated' for r in rows),
          sum(r['ai_status']=='unavailable' for r in rows),
          sum(not r['prediction_created_at'] for r in rows))


## 実行済みの読み取り専用D1クエリ
Wrangler d1 execute --remote --env preview --command を使用。資格情報や監査ログ中の組織識別子はこのノートに含めない。
```sql
SELECT id,run_slot,run_at,publish_at,completed_at,status,target_count,success_count,error_count FROM forecast_runs ORDER BY run_slot DESC LIMIT 5;
SELECT a.ai_status,a.error_message,COUNT(*) AS count FROM ai_predictions a WHERE a.forecast_run_id=(SELECT id FROM forecast_runs WHERE status IN ('completed','partial') ORDER BY run_slot DESC LIMIT 1) GROUP BY a.ai_status,a.error_message;
SELECT message FROM run_logs WHERE event='ai_batch' AND forecast_run_id=(SELECT id FROM forecast_runs ORDER BY run_slot DESC LIMIT 1);
```
最新実行: 対象31、保存31、AI生成28、AI unavailable3。完了15:55:58 JST、予定公開16:00 JST。
主系Groq20bは便漏れ/JSON不備/429で失敗し、120b呼び出しまで到達せずGeminiへ切替。最後の3便はGemini429とWorker subrequest上限により評価が生成されなかった。
10/1 08:35東京発Cと10:40大島発Aは大島側波高がともに1.68m。入力では往路にだけ大島入港1mの基準比較を生成している。
